### Importing necessary modules

In [174]:
import pandas as pd                                                                     # type: ignore
import matplotlib.pyplot as plt                                                         # type: ignore
import seaborn as sns                                                                   # type: ignore
import numpy as np                                                                      # type: ignore
import tensorflow as tf                                                                 # type: ignore
import matplotlib as mpl                                                                # type: ignore
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
mpl.rcParams['figure.figsize'] = (9, 7)
from joblib import dump, load                                                           # type: ignore
import random                                                                           # type: ignore
import cartopy.crs as ccrs                                                              # type: ignore
import cartopy.feature as cfeature                                                      # type: ignore
#np.random.seed(42)                                      
from netCDF4 import Dataset                                                             # type: ignore

In [175]:
geodata = Dataset("/home/mendrika/mendrika-phd/codes/nflics/geoloc_grids/nxny1640_580_nxnyds164580_blobdx0.04491576_area4_n23_20_32.nc")

Bamako_lon = -8.0029
Bamako_lat = 12.6392

In [176]:
step = 0.1

longitude = np.arange(-12, 0, step)
latitude = np.arange(10, 18, step)

lons, lats = np.meshgrid(longitude, latitude)

In [177]:
import sys
sys.path.insert(1, "/home/mendrika/mendrika-phd/codes/nflics")
import nflics  

### Importing dataset

### Exploratory data analysis

In [178]:
def log_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i > 0:
                transformed.append(np.log(i))
            else:
                transformed.append(np.log(i+1e-8))    
        df_copy[key] = transformed
    return df_copy

In [179]:
def sqrt_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i >= 0:
                transformed.append(np.sqrt(i)) 
        df_copy[key] = transformed
    return df_copy

In [180]:
# Presence or absence of convection in Dakar at time t+1
target_index = "Cb_Dakar"

# Input at time t0
t0 = "year,month,day,hour,minute,"

# latitude and longitude, wavelet power, storm size and distance to Dakar
location = ""
wavelet_power = ""
storm_size = ""
distance = ""
for i in range(1,6):
    location += f"lat{i},lon{i},"
    wavelet_power += f"wp{i},"
    distance += f"ds{i},"
    storm_size += f"size{i},"

# combining all fields to form the feature input
features = t0 + location + wavelet_power + storm_size + distance
field =  features + target_index
field = field.split(',')

In [181]:
to_scale = wavelet_power + storm_size + distance

In [182]:
model_t1 = tf.keras.models.load_model("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/lead-time/Bamako-t1.keras")

In [183]:
scaler = load('/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/lead-time/scaler-Bamako.bin')

Computing the average wavelet power

In [184]:
mean_wp = 3430.40
median_wp = 2266.0

mean_size = 7460.30
median_size = 5000.0

In [185]:
def create_storm():
    return {
        'year': [], 
        'month': [], 
        'day': [], 
        'hour': [], 
        'minute': [], 
        'lat1': [], 
        'lon1': [], 
        'lat2': [], 
        'lon2': [], 
        'lat3': [], 
        'lon3': [], 
        'lat4': [], 
        'lon4': [], 
        'lat5': [], 
        'lon5': [], 
        'wp1': [], 
        'wp2': [], 
        'wp3': [], 
        'wp4': [], 
        'wp5': [], 
        'size1': [], 
        'size2': [], 
        'size3': [], 
        'size4': [], 
        'size5': [], 
        'ds1': [], 
        'ds2': [], 
        'ds3': [], 
        'ds4': [], 
        'ds5': [], 
    }

For reproducibility

In [186]:
rng1 = np.random.RandomState(26)
rng2 = np.random.RandomState(21)
def generate_geo_coords(lat_min, lat_max, lon_min, lon_max):
    artificial_storm_lat = round(rng1.uniform(lat_min, lat_max), 6)
    artificial_storm_lon = round(rng1.uniform(lon_min, lon_max), 6)
    return artificial_storm_lat, artificial_storm_lon

In [187]:
def distance_to_Bamako(lat, lon):
    Bamako_lon = -8.0029
    Bamako_lat = 12.6392
    Bamako_y, Bamako_x = nflics.X0(Bamako_lat, Bamako_lon)
    y, x = nflics.X0(lat, lon)
    return round(np.sqrt((y - Bamako_y)**2 + (x - Bamako_x)**2), 4)

In [188]:
number_of_data_points = 1
number_of_storms_to_generate = 1
number_of_storms_in_model = 5

In [189]:
def generate_storm_cluster_from_a_center(lat_center, lon_center):    
    lat_clusters = []
    lon_clusters = []    
    for _ in range(5):
        lat_candidate = lat_center + random.normalvariate(0,0.2)    
        lat_clusters.append(lat_candidate)    
        lon_candidate = lon_center + random.normalvariate(0,0.2)
        lon_clusters.append(lon_candidate)
    return lat_clusters, lon_clusters

In [190]:
artificial_storms = create_storm()
for lat_center, lon_center in zip(lats.ravel()[:], lons.ravel()[:]):
    for _ in range(number_of_data_points):
        year, month, day, hour, minute = 2020, 7, 15, 18, 0
        artificial_storms["year"].append(year)
        artificial_storms["month"].append(month)
        artificial_storms["day"].append(day)
        artificial_storms["hour"].append(hour)
        artificial_storms["minute"].append(minute)
        lat_candidates, lon_candidates = generate_storm_cluster_from_a_center(lat_center, lon_center)
        for j in range(1, number_of_storms_in_model + 1):                        
            artificial_storm_lat, artificial_storm_lon = lat_candidates[j-1], lon_candidates[j-1]  #field name indices start from 1
            artificial_storms[f"lat{j}"].append(artificial_storm_lat)
            artificial_storms[f"lon{j}"].append(artificial_storm_lon)
            d_to_Bamako = distance_to_Bamako(artificial_storm_lat, artificial_storm_lon)
            artificial_storms[f"ds{j}"].append(d_to_Bamako)
            artificial_storms[f"wp{j}"].append(median_wp)
            artificial_storms[f"size{j}"].append(median_size)        

raw_artificial_storm_data = pd.DataFrame.from_dict(artificial_storms, orient='columns')
artificial_storm_data = log_transform(raw_artificial_storm_data, to_scale.split(',')[:-1])
artificial_storm_data = sqrt_transform(artificial_storm_data, t0.split(',')[:-1])

raw_artificial_storm_data.to_csv("test-data-from-latlon-map-Bamako.csv", index=False)